# eeg2img on Colab: training

Runs a queue of experiments back to back on the store built by `colab_prepare_data.ipynb`
(same session, or a copy of the store). Each job resumes if interrupted and is skipped once
finished. T4 wall time: ridge 3 min, EEGNet 18 min, CNN 20 min, transformer head 35 min.


In [ ]:
REPO = "moh246/eeg2image-neurips26"
import os, subprocess
if not os.path.isdir("/content/eeg2img"):
    url = f"https://github.com/{REPO}.git"
    try:  # only needed while the repository is private
        from google.colab import userdata
        url = f"https://{userdata.get('GH_TOKEN')}@github.com/{REPO}.git"
    except Exception:
        pass
    subprocess.run(["git", "clone", "--quiet", url, "/content/eeg2img"], check=True)
%cd /content/eeg2img
!git log -1 --format='%h %s'
%pip install --quiet -e ".[data]"


In [ ]:
STORE = "/content/work/stores/alljoined16m_v1"
RUNS = "/content/runs"

!bash scripts/run_queue.sh {RUNS} \
    alljoined_ridge:0 alljoined_eegnet:0 \
    alljoined_conv:0 alljoined_conv:1 alljoined_conv:2 \
    alljoined_conv_transformer:0 alljoined_conv_transformer:1 alljoined_conv_transformer:2 \
    alljoined_conv_transformer_causal:0 \
    -- data.store={STORE} data.embeddings={STORE}/embeddings.npy


## Results table and run records


In [ ]:
!python scripts/summarize_runs.py --registry {RUNS}/experiments/registry.jsonl

# Archive the run records; import them into the repository with scripts/import_runs.py.
import shutil
from google.colab import files
files.download(shutil.make_archive("/content/eeg2img_runs", "gztar", RUNS, "experiments"))


## Follow-ups

Seed ensembles, the electrode-removal test and the longer transformer schedule. Add the
remaining seeds and `alljoined_channel_transformer:0` to the queue above first if you want
them included.


In [ ]:
!bash scripts/colab_followups.sh {RUNS} -- data.store={STORE} data.embeddings={STORE}/embeddings.npy
